# 01｜Shiller CAPE 與長期市場報酬預測
**資料來源：Robert Shiller（耶魯大學）— 2013 年諾貝爾經濟學獎得主**

> **核心問題：股市現在貴不貴？怎麼用資料判斷？**
>
> 本 Notebook 使用 Shiller 的 150 年歷史資料，驗證一個簡單但強大的發現：
> **當股市估值（CAPE）偏高時，未來 10 年的實質報酬率往往偏低。**

## 你覺得現在的股市貴嗎？

這個問題很難回答，因為「貴」是相對的。

2021 年底台積電漲到接近 700 元，很多人說「太貴了不敢買」。
2022 年底跌到 370 元，又有人說「便宜了，現在可以進場」。
2024 年後漲到四位數——那到底是貴還是便宜？

**光看股價本身沒有意義。** 一支 100 元的股票可能比 50 元的更便宜，
因為關鍵不是「它現在多少錢」，而是「它每年賺多少錢、值不值這個價」。

這就是「估值」要解決的問題：相對於獲利能力，現在的股價合理嗎？

Shiller 教授（2013 年諾貝爾獎得主）發明了一個比普通 P/E 更穩定的指標——
**CAPE（週期性調整本益比）**：用過去 10 年的平均獲利來計算，
消除一時景氣好壞造成的數字扭曲。

這一章，我們用 150 年的歷史數據來驗證：**CAPE 有沒有辦法預測未來的股市報酬？**

## 🎯 學習目標
1. 理解「本益比平均化（CAPE）」的計算邏輯與歷史意義
2. 用散點圖驗證「CAPE 高 → 未來10年報酬低」的統計關係
3. 根據當前 CAPE 水準，推估合理的長期報酬預期

## ⚙️ Step 0：安裝套件

In [ ]:
%pip install xlrd openpyxl
print("✅ 完成")

## 匯入套件

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['font.family'] = ['Microsoft YaHei', 'SimHei', 'Heiti TC', 'PingFang HK', 'STHeiti', 'Arial Unicode MS', 'sans-serif']
matplotlib.rcParams['axes.unicode_minus'] = False
print("✅ 匯入完成")

## 一、什麼是 CAPE？

### 普通 P/E 的問題
企業盈餘（EPS）隨景氣循環大幅波動：
- 景氣好 → EPS 高 → P/E 看起來低（「很便宜！」→ 其實在高點）
- 景氣差 → EPS 崩潰 → P/E 突然很高（「好貴！」→ 其實在低點）

用普通 P/E 判斷時機，容易得出錯誤結論。

### CAPE 的解法（Cyclically Adjusted P/E）
> **CAPE = 股價 ÷ 過去 10 年平均實質盈餘（剔除通膨）**

用 10 年平均平滑掉景氣循環的雜訊，讓估值更有參考意義。

| | 普通 P/E | CAPE |
|--|---------|------|
| 盈餘基準 | 當年 EPS | 過去 10 年平均實質 EPS |
| 景氣敏感度 | 高（容易誤導） | 低（更穩定） |
| 預測能力 | 差 | 對未來 10 年報酬有顯著預測力 |
| 歷史均值 | 約 15–16 倍 | 約 16–17 倍 |

## 二、載入 Shiller 資料

In [ ]:
import os

LOCAL = "data/shiller_data.csv"

if os.path.exists(LOCAL):
    df = pd.read_csv(LOCAL, parse_dates=['date'], index_col='date')
    print(f"✅ 從本機載入（{len(df)} 筆）")
else:
    print("⏳ 從網路下載 Shiller 資料...")
    url = "http://www.econ.yale.edu/~shiller/data/ie_data.xls"
    raw = pd.read_excel(url, sheet_name="Data", skiprows=7, header=0)

    # 過濾有效資料列
    raw = raw[pd.to_numeric(raw.iloc[:, 0], errors='coerce').notna()].copy()
    raw = raw[raw.iloc[:, 0].astype(float) > 1800].copy()

    # 解析十進位日期：1871.01 = 1月，1871.10 = 10月
    dec = raw.iloc[:, 0].astype(float)
    year  = dec.apply(lambda x: int(x))
    month = dec.apply(lambda x: max(1, int(round((x % 1) * 100))))

    df = pd.DataFrame({
        'date'       : pd.to_datetime({'year': year, 'month': month, 'day': 1}),
        'price'      : pd.to_numeric(raw.iloc[:, 1],  errors='coerce'),
        'dividend'   : pd.to_numeric(raw.iloc[:, 2],  errors='coerce'),
        'earnings'   : pd.to_numeric(raw.iloc[:, 3],  errors='coerce'),
        'cpi'        : pd.to_numeric(raw.iloc[:, 4],  errors='coerce'),
        'long_rate'  : pd.to_numeric(raw.iloc[:, 6],  errors='coerce'),
        'real_price' : pd.to_numeric(raw.iloc[:, 7],  errors='coerce'),
        'cape'       : pd.to_numeric(raw.iloc[:, 12], errors='coerce'),
    }).dropna(subset=['date', 'price', 'cape'])

    df = df.set_index('date').sort_index()
    df.to_csv(LOCAL)
    print(f"✅ 下載完成，已快取到 {LOCAL}（{len(df)} 筆）")

df.tail(3)

In [ ]:
print(f"資料期間：{df.index[0].strftime('%Y-%m')} 至 {df.index[-1].strftime('%Y-%m')}")
print(f"總月數　：{len(df)} 個月（約 {len(df)//12} 年）")
print(f"CAPE 歷史均值  ：{df['cape'].mean():.1f} 倍")
print(f"CAPE 歷史中位數：{df['cape'].median():.1f} 倍")
print(f"CAPE 最高點：{df['cape'].max():.1f} 倍（{df['cape'].idxmax().strftime('%Y-%m')}）")
print(f"CAPE 最低點：{df['cape'].min():.1f} 倍（{df['cape'].idxmin().strftime('%Y-%m')}）")

## 三、S&P 500 — 150 年走勢

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.semilogy(df.index, df['real_price'], color='steelblue', linewidth=0.7)

events = {
    '1929\n大崩盤': ('1929-09', 2.5),
    '1966\n高峰':   ('1966-02', 2.8),
    '2000\n科技泡沫':('2000-08', 2.5),
    '2008\n金融海嘯':('2007-10', 2.5),
}
for label, (date_str, mult) in events.items():
    date = pd.to_datetime(date_str)
    y = df['real_price'].asof(date)
    ax.annotate(label, xy=(date, y), xytext=(date, y * mult),
                arrowprops=dict(arrowstyle='->', color='red', lw=1),
                fontsize=8, ha='center', color='darkred')

ax.set_title('S&P 500 實質價格走勢（1871–今，對數尺度）', fontsize=13)
ax.set_ylabel('實質價格指數（對數）')
ax.set_xlabel('年份')
plt.tight_layout()
plt.show()

## 四、CAPE 歷史走勢

In [ ]:
cape_mean = df['cape'].mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df['cape'], color='darkorange', linewidth=0.7, label='CAPE')
ax.axhline(cape_mean, color='gray', linestyle='--', linewidth=1.2,
           label=f'歷史均值 ({cape_mean:.1f}x)')
ax.axhline(25, color='red', linestyle=':', linewidth=1, alpha=0.6, label='警戒線 (25x)')

ax.fill_between(df.index, cape_mean, df['cape'],
                where=(df['cape'] > cape_mean), alpha=0.12, color='red', label='高於均值（偏貴）')
ax.fill_between(df.index, df['cape'], cape_mean,
                where=(df['cape'] < cape_mean), alpha=0.12, color='green', label='低於均值（偏便宜）')

peaks = {'1929\n(31x)': '1929-09', '2000\n(44x)': '1999-12', '2022\n(38x)': '2021-10'}
for label, d in peaks.items():
    date = pd.to_datetime(d)
    y = df['cape'].asof(date)
    ax.annotate(label, xy=(date, y), xytext=(date, y + 4),
                fontsize=8, ha='center', color='darkred',
                arrowprops=dict(arrowstyle='->', color='darkred', lw=0.8))

ax.set_title('Shiller CAPE 歷史走勢（1881–今）', fontsize=13)
ax.set_ylabel('CAPE（倍）')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## 五、核心驗證：CAPE 能預測未來報酬嗎？

**假說：CAPE 越高，未來 10 年實質報酬越低。**

我們把每個月的 CAPE 值，對應到「該月買入、持有 10 年的年化實質報酬率」，
用散點圖直接驗證兩者的關係。

In [ ]:
# 計算未來 10 年年化實質報酬率（120 個月後）
df['real_price_10y'] = df['real_price'].shift(-120)
df['return_10y'] = (df['real_price_10y'] / df['real_price']) ** (1/10) - 1

test = df.dropna(subset=['cape', 'return_10y']).copy()

fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(test['cape'], test['return_10y'] * 100,
                c=test.index.year, cmap='plasma', alpha=0.35, s=8)
plt.colorbar(sc, ax=ax, label='年份')

# 趨勢線
z = np.polyfit(test['cape'], test['return_10y'] * 100, 1)
p = np.poly1d(z)
xr = np.linspace(test['cape'].min(), test['cape'].max(), 200)
ax.plot(xr, p(xr), 'r-', linewidth=2, label='趨勢線')

corr = test['cape'].corr(test['return_10y'])
ax.set_title(f'CAPE vs 未來 10 年實質年化報酬率\n（相關係數 = {corr:.2f}）', fontsize=13)
ax.set_xlabel('當時的 CAPE（倍）')
ax.set_ylabel('未來 10 年實質年化報酬率（%）')
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 按 CAPE 五分位數分組，看各組的平均未來報酬
test['cape_group'] = pd.qcut(test['cape'], q=5,
    labels=['最低 20%\n（最便宜）', '次低 20%', '中間 20%', '次高 20%', '最高 20%\n（最貴）'])

summary = test.groupby('cape_group', observed=True)['return_10y'].agg(['mean', 'std']) * 100
summary.columns = ['平均年化報酬 (%)', '標準差 (%)']
print("CAPE 分位數 vs 未來 10 年年化實質報酬：")
print(summary.round(2).to_string())

## 六、討論

**結論：CAPE 與未來 10 年報酬呈顯著負相關（相關係數約 -0.6 至 -0.7）。**

這不是在說「CAPE 高，明天就會崩」，而是：
> 「以目前價格買進，未來 10 年的期望年化報酬率比長期均值低。」

**思考問題：**
1. 目前 CAPE 約 30+ 倍。歷史上僅 1929、2000 年前後曾達到此水位。之後分別發生了什麼？
2. 分組統計裡，「最便宜 20%」和「最貴 20%」的報酬差距有多大？這在投資實務上意味著什麼？
3. 低 CAPE ≠ 立刻上漲。有沒有 CAPE 低但接下來十年報酬仍然不好的時期？為什麼？
4. 台股的 CAPE 歷史均值和美股相同嗎？如果不同，為什麼？

## 七、這跟你有什麼關係？

**CAPE 是估值溫度計，不是擇時工具。**

你剛才建立的散點圖回歸線，可以直接用來推算：
「以目前估值買入，未來 10 年的期望報酬大概是多少？」

這不能告訴你「明年漲不漲」，但可以幫你校準長期報酬預期，
避免在高點把 All-in 的獲利預期設得不切實際。

In [ ]:
# p 是上方散點圖建立的回歸線，直接代入現在的 CAPE
current_cape  = df['cape'].iloc[-1]
predicted_10y = p(current_cape)

print(f"目前 S&P 500 CAPE：{current_cape:.1f} 倍（歷史均值 {df['cape'].mean():.1f} 倍）")
print(f"根據 145 年歷史回歸，預測未來 10 年實質年化報酬：約 {predicted_10y:.1f}%")
print()

# 各估值水位的預測
print(f"{'CAPE':>8} | {'預期10年報酬':>12}")
print("-" * 25)
for cape_val in [15, 20, 25, 30, 35, 40, round(current_cape)]:
    ret = p(cape_val)
    marker = " ← 目前" if abs(cape_val - current_cape) < 2 else ""
    print(f"{cape_val:>8.0f} | {ret:>+11.1f}%{marker}")

print()
print("重點：CAPE 在 30 倍以上時，過去 145 年的期望報酬多數落在 0–4%")
print("這不是說『不要投資』，而是：")
print("  → 降低期望值，把計畫報酬率從 7% 調成 4–5%")
print("  → 不要在高估值時一次 All-in，分批更能降低序列風險")
print("  → 費用控制和持續投資比擇時更重要")

## 📌 本章重點摘要
| 概念 | 結論 |
|------|------|
| CAPE 與未來報酬 | 負相關（r ≈ −0.8），CAPE 高 → 未來10年報酬偏低 |
| 均值迴歸 | 歷史 CAPE 均值 ≈ 16；超過30時需謹慎 |
| 個人應用 | CAPE 適合判斷長期報酬預期，不適合短線擇時 |

> **下一章：** Damodaran 行業估值數據庫，看懂不同行業的合理 P/E